In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
!pip install ultralytics

In [ ]:
!unzip -q /content/drive/MyDrive/EdgeCard_System/dataset_yolo_obb.zip

In [ ]:
import yaml
from ultralytics import YOLO

# # 1. Tạo cấu hình data.yaml cho tập dữ liệu OBB
# data_config = {
#     'path': '/content/dataset_yolo_obb',
#     'train': 'images/train',
#     'val': 'images/val',
#     'names': {0: 'text_field'}
# }

# with open('dataset_yolo_obb.yaml', 'w') as f:
#     yaml.dump(data_config, f)

# 2. Khởi tạo mô hình bản OBB (nhớ gắn hậu tố -obb vào tệp trọng số)
model = YOLO('yolo26n-obb.pt')

# 3. Huấn luyện với task='obb'
results = model.train(
    task='obb',          # Bắt buộc khai báo task OBB
    data='/content/dataset_yolo_obb/data.yaml',
    project='/content/drive/MyDrive/EdgeCard_System/stage_2/yolo26n-obb',
    name='Phase2_TextDet_OBB',

    # --- Cấu hình Huấn luyện Lõi ---
    epochs=50,
    batch=32,            # Giữ nguyên batch 32 để tận dụng T4
    imgsz=640,
    device=0,

    # --- Cấu hình Tối ưu hóa (Optimizer) ---
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,

    # --- Đánh giá & Lưu trữ ---
    val=True,
    save=True,

    # --- Tăng cường Dữ liệu (Dành riêng cho OBB) ---
    fliplr=0.0,          # TẮT lật ngang (cực kỳ quan trọng với text)
    flipud=0.0,          # TẮT lật dọc
    degrees=10.0,         # Chỉ xoay nhẹ 10 độ
    scale=0.2
)

print(f"Độ chính xác mAP@50 đạt: {results.box.map50:.3f}")

In [ ]:
print('Đang tiến hành đánh giá mô hình trên tập test...')

# Đánh giá mô hình trên tập test
metrics = model.val(
    task='obb',          # Bắt buộc khai báo task OBB
    data='/content/dataset_yolo_obb/data.yaml', # Đường dẫn đến file cấu hình dataset
    project='/content/drive/MyDrive/EdgeCard_System/stage_2/yolo26n-obb', # Project để lưu kết quả
    name='Phase2_TextDet_OBB_Test_Evaluation', # Tên của lần chạy đánh giá
    imgsz=640,
    device=0,
    split='test'         # Chỉ định đánh giá trên tập test
)

print(f"Kết quả đánh giá trên tập test:")
print(f"  mAP50: {metrics.box.map50:.3f}")
print(f"  mAP50-95: {metrics.box.map:.3f}")
print("Đánh giá hoàn tất.")